# Konkani Companion — Colab backend runner

Runs the backend in `backend/app` on a Colab GPU and exposes it through ngrok.
No keys live in this notebook.

**One-time setup:** open the 🔑 *Secrets* panel on the left and add:

| Secret | Required | Notes |
|---|---|---|
| `GROQ_API_KEY` | yes | Reasoning model |
| `BACKEND_API_KEY` | yes | Shared secret; put the same value in the frontend's `.env.local` |
| `NGROK_AUTHTOKEN` | yes | ngrok tunnel |
| `NGROK_DOMAIN` | no | Your reserved ngrok domain, so the URL never changes |
| `FIRECRAWL_API_KEY` | no | Primary web search |
| `TAVILY_API_KEY` | no | Fallback web search (DuckDuckGo is the last resort, no key needed) |
| `GITHUB_TOKEN` | only if the repo is private | Read-only token for cloning |

Enable notebook access for each secret. Runtime → Change runtime type → **T4 GPU**.

Run cells 1–4 once per runtime. Re-run only cell 5 to restart the server (models stay loaded).

In [ ]:
# 1. Get the code
import os, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/boybothere/konkani_companion.git"
BRANCH = "main"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if os.path.isdir("/content/kc/.git"):
    subprocess.run(["git", "-C", "/content/kc", "pull", "--ff-only"], check=True)
else:
    token = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/kc"], check=True)
    subprocess.run(["git", "-C", "/content/kc", "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", "/content/kc", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install system packages (OCR + audio decoding) and Python dependencies
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-mar ffmpeg > /dev/null
!pip install -q unsloth
!pip install -q -r /content/kc/backend/requirements.txt -r /content/kc/backend/requirements-gpu.txt pyngrok
!tesseract --list-langs
# If pip asks you to restart the runtime, do it, then continue from cell 3.

In [ ]:
# 3. Load secrets into the environment (values are never printed)
import os
from google.colab import userdata

NAMES = ["GROQ_API_KEY", "BACKEND_API_KEY", "NGROK_AUTHTOKEN", "NGROK_DOMAIN",
         "FIRECRAWL_API_KEY", "TAVILY_API_KEY"]
for name in NAMES:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
print({name: ("set" if os.environ.get(name) else "missing") for name in NAMES})
for required in ("GROQ_API_KEY", "BACKEND_API_KEY", "NGROK_AUTHTOKEN"):
    assert os.environ.get(required), f"Add the {required} secret first."

# Optional overrides (see backend/.env.example for all settings)
# os.environ["WHISPER_MODEL"] = "large-v3"
# os.environ["SEARCH_PROVIDERS"] = "tavily,duckduckgo"

In [ ]:
# 4. Load the models once (Qwen LoRA + Whisper). Takes a few minutes.
import sys, logging
sys.path.insert(0, "/content/kc/backend")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s", force=True)

from app.config import get_settings
from app.pipeline import build_services

services = build_services(get_settings())
print("Models loaded. Search providers:", services.search.configured_providers())

In [ ]:
# 5. Start the API and the ngrok tunnel. Interrupt this cell to stop the server.
import os
import uvicorn
from pyngrok import ngrok
from app.main import create_app

app = create_app(services)
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
ngrok.kill()
options = {"domain": os.environ["NGROK_DOMAIN"]} if os.environ.get("NGROK_DOMAIN") else {}
tunnel = ngrok.connect(8000, **options)
print("PUBLIC URL:", tunnel.public_url)
print("Set BACKEND_URL to this in the frontend's .env.local")

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
try:
    await server.serve()
finally:
    ngrok.kill()